In [2]:
import pandas as pd
import numpy as np
import re
import json
import time
import pickle
from tqdm import tqdm
from nltk.stem.snowball import RussianStemmer
import bm25s

tqdm.pandas()

C:\Users\googl\python_global_env\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Данные
benchmark_items = pd.read_parquet('../dataset/benchmark_items.parquet')
benchmark_queries = pd.read_parquet('../dataset/benchmark_queries.parquet')
benchmark_items['item_id'] = benchmark_items['item_id'].astype(str)
benchmark_queries['query_id'] = benchmark_queries['query_id'].astype(str)

print("benchmark_items:", benchmark_items.shape)
print("benchmark_queries:", benchmark_queries.shape)

# Модель v2
with open('../dataset/lgbm_reranker_v2.pkl', 'rb') as f:
    payload = pickle.load(f)
final_model = payload['model']
feature_cols = payload['feature_cols']
print(f"Модель v2 загружена, {len(feature_cols)} признаков")
print("Порядок признаков:")
for i, c in enumerate(feature_cols):
    print(f"  {i}: {c}")

benchmark_items: (189212, 14)
benchmark_queries: (2452, 6)
Модель v2 загружена, 20 признаков
Порядок признаков:
  0: bm25_full_score
  1: bm25_full_rank
  2: bm25_title_score
  3: bm25_title_rank
  4: bm25_char3_score
  5: bm25_char3_rank
  6: overlap_recall
  7: overlap_jaccard
  8: title_overlap_recall
  9: desc_overlap_recall
  10: params_overlap_recall
  11: char3_jaccard
  12: loc_match
  13: cat_match
  14: microcat_match
  15: item_rating
  16: item_reviews
  17: item_price_log
  18: item_title_len
  19: item_desc_len


In [4]:
r_full = bm25s.BM25.load('../dataset/bm25_index', load_corpus=False)
r_title = bm25s.BM25.load('../dataset/bm25_index_title', load_corpus=False)
r_char3 = bm25s.BM25.load('../dataset/bm25_index_char3', load_corpus=False)
print("BM25 индексы загружены")

BM25 индексы загружены


In [5]:
tokens_by_id = {}
with open('../dataset/item_tokens_fields.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        obj = json.loads(line)
        tokens_by_id[obj['item_id']] = obj

item_char3_by_id = {}
with open('../dataset/item_title_char3.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        obj = json.loads(line)
        item_char3_by_id[obj['item_id']] = set(obj['tokens'])

print(f"Токены: {len(tokens_by_id)}, char3: {len(item_char3_by_id)}")

Токены: 189212, char3: 189212


In [6]:
stemmer = RussianStemmer()
TOKEN_RE = re.compile(r'[а-яa-z0-9]+')
_STEM_CACHE = {}

def tokenize_stem(text):
    if not isinstance(text, str) or not text:
        return []
    text = text.lower().replace('ё', 'е')
    tokens = TOKEN_RE.findall(text)
    out = []
    for t in tokens:
        if len(t) <= 1:
            continue
        s = _STEM_CACHE.get(t)
        if s is None:
            s = stemmer.stem(t)
            _STEM_CACHE[t] = s
        out.append(s)
    return out

def char_3gram_tokens(text):
    if not isinstance(text, str) or not text:
        return []
    text = text.lower().replace('ё', 'е')
    text = re.sub(r'[^а-яa-z0-9]', '', text)
    if len(text) < 3:
        return [text] if text else []
    return [text[i:i+3] for i in range(len(text) - 2)]

benchmark_queries['q_tokens'] = [tokenize_stem(t) for t in tqdm(
    benchmark_queries['search_query'].tolist(), desc="tok queries")]
benchmark_queries['q_char3'] = [char_3gram_tokens(t) for t in tqdm(
    benchmark_queries['search_query'].tolist(), desc="char3 queries")]

print("Пример:", benchmark_queries['search_query'].iloc[0])

char3 queries: 100%|██████████| 2452/2452 [00:00<00:00, 98886.88it/s]

Пример: перевозки владикавказ тбилиси


In [7]:
K_BM25 = 1000
BATCH = 128

def batch_bm25(retriever, q_tokens_list, k, name):
    all_ids = [None] * len(q_tokens_list)
    all_scores = [None] * len(q_tokens_list)
    for s in tqdm(range(0, len(q_tokens_list), BATCH), desc=f"BM25 {name}"):
        e = min(s + BATCH, len(q_tokens_list))
        batch_q = q_tokens_list[s:e]
        safe = [q if q else ['_empty_'] for q in batch_q]
        res, sc = retriever.retrieve(safe, k=k, show_progress=False)
        for j in range(len(batch_q)):
            all_ids[s + j] = res[j]
            all_scores[s + j] = sc[j]
    return all_ids, all_scores

q_tokens_list = benchmark_queries['q_tokens'].tolist()
q_char3_list = benchmark_queries['q_char3'].tolist()

bm25_full_ids, bm25_full_scores = batch_bm25(r_full, q_tokens_list, K_BM25, "full")
bm25_title_ids, bm25_title_scores = batch_bm25(r_title, q_tokens_list, K_BM25, "title")
bm25_char3_ids, bm25_char3_scores = batch_bm25(r_char3, q_char3_list, K_BM25, "char3")
print("BM25 поиск завершён")

BM25 char3: 100%|██████████| 20/20 [00:06<00:00,  3.33it/s]

BM25 поиск завершён


In [8]:
def to_float(s):
    if pd.isna(s): return np.nan
    s = str(s).replace(',', '.').replace(' ', '')
    try: return float(s)
    except: return np.nan

benchmark_items['item_price'] = benchmark_items['item_price'].apply(to_float)
benchmark_items['item_rating'] = pd.to_numeric(benchmark_items['item_rating'], errors='coerce').fillna(-1)
benchmark_items['item_rating_reviews_count'] = pd.to_numeric(
    benchmark_items['item_rating_reviews_count'], errors='coerce').fillna(-1)

bi = benchmark_items.reset_index(drop=True)

item_ids_array = bi['item_id'].values
item_microcat_arr = bi['item_microcat_id'].values
item_loc_arr = bi['item_location_id'].values
item_rating_arr = bi['item_rating'].values
item_reviews_arr = bi['item_rating_reviews_count'].values
item_price_log_arr = np.log1p(bi['item_price'].fillna(0).clip(lower=0).values)
item_title_len_arr = bi['item_title_raw'].fillna('').astype(str).str.len().values
item_desc_len_arr = bi['item_description_raw'].fillna('').astype(str).str.len().values

item_tokens_title_list = [tokens_by_id.get(iid, {}).get('title', []) for iid in item_ids_array]
item_tokens_desc_list = [tokens_by_id.get(iid, {}).get('desc', []) for iid in item_ids_array]
item_tokens_params_list = [tokens_by_id.get(iid, {}).get('params', []) for iid in item_ids_array]
item_char3_sets = [item_char3_by_id.get(iid, set()) for iid in item_ids_array]

print("Массивы готовы")

Массивы готовы


In [9]:
# Обратные лукапы: {item_pos: rank}
rank_full = [{int(ip): r for r, ip in enumerate(arr)} for arr in tqdm(bm25_full_ids, desc="rank_full")]
rank_title = [{int(ip): r for r, ip in enumerate(arr)} for arr in tqdm(bm25_title_ids, desc="rank_title")]
rank_char3 = [{int(ip): r for r, ip in enumerate(arr)} for arr in tqdm(bm25_char3_ids, desc="rank_char3")]

# Множества токенов для item'ов — один раз
item_set_title = [set(t) for t in tqdm(item_tokens_title_list, desc="set_title")]
item_set_desc = [set(t) for t in tqdm(item_tokens_desc_list, desc="set_desc")]
item_set_params = [set(t) for t in tqdm(item_tokens_params_list, desc="set_params")]
item_set_all = [item_set_title[i] | item_set_desc[i] | item_set_params[i]
                for i in range(len(item_ids_array))]

print("Предвычисления готовы")

set_params: 100%|██████████| 189212/189212 [00:01<00:00, 136140.19it/s]


Предвычисления готовы


In [10]:
q_loc_arr = benchmark_queries['search_location_id'].astype(str).tolist()
q_cat_arr = benchmark_queries['search_category'].astype(str).tolist()
q_char3_arr = benchmark_queries['q_char3'].tolist()

def compute_features_for_query_fast(qp):
    pool = set(bm25_full_ids[qp].tolist()) | \
           set(bm25_title_ids[qp].tolist()) | \
           set(bm25_char3_ids[qp].tolist())
    item_pos_list = list(pool)
    n = len(item_pos_list)
    X = np.zeros((n, len(feature_cols)), dtype=np.float32)

    qt = q_tokens_list[qp]
    qs = set(qt)
    q_char3 = set(q_char3_arr[qp])
    q_loc = q_loc_arr[qp]
    q_cat = q_cat_arr[qp]

    qs_len = len(qs) if qs else 1
    qc3_len = len(q_char3) if q_char3 else 1

    lf = rank_full[qp]; lt = rank_title[qp]; lc3 = rank_char3[qp]
    sf_arr = bm25_full_scores[qp]
    st_arr = bm25_title_scores[qp]
    sc3_arr = bm25_char3_scores[qp]

    for i, ip in enumerate(item_pos_list):
        at = item_set_all[ip]
        ts = item_set_title[ip]
        ds = item_set_desc[ip]
        ps = item_set_params[ip]

        rf = lf.get(ip, 9999); sf = float(sf_arr[rf]) if rf < 9999 else 0.0
        rt = lt.get(ip, 9999); st = float(st_arr[rt]) if rt < 9999 else 0.0
        rc3 = lc3.get(ip, 9999); sc3 = float(sc3_arr[rc3]) if rc3 < 9999 else 0.0

        if qs and at:
            inter = len(qs & at)
            ovr = inter / qs_len
            denom = qs_len + len(at) - inter
            ovj = inter / denom if denom > 0 else 0.0
        else:
            ovr = ovj = 0.0

        tor = len(qs & ts) / qs_len if qs else 0.0
        dor = len(qs & ds) / qs_len if qs else 0.0
        por = len(qs & ps) / qs_len if qs else 0.0

        item_c3 = item_char3_sets[ip]
        if q_char3 and item_c3:
            inter_c3 = len(q_char3 & item_c3)
            denom_c3 = qc3_len + len(item_c3) - inter_c3
            c3 = inter_c3 / denom_c3 if denom_c3 > 0 else 0.0
        else:
            c3 = 0.0

        X[i, 0] = sf
        X[i, 1] = rf
        X[i, 2] = st
        X[i, 3] = rt
        X[i, 4] = sc3
        X[i, 5] = rc3
        X[i, 6] = ovr
        X[i, 7] = ovj
        X[i, 8] = tor
        X[i, 9] = dor
        X[i, 10] = por
        X[i, 11] = c3
        X[i, 12] = 1 if q_loc == str(item_loc_arr[ip]) else 0
        X[i, 13] = 1 if q_cat == str(item_microcat_arr[ip]) else 0
        X[i, 14] = X[i, 13]
        X[i, 15] = float(item_rating_arr[ip])
        X[i, 16] = float(item_reviews_arr[ip])
        X[i, 17] = float(item_price_log_arr[ip])
        X[i, 18] = float(item_title_len_arr[ip])
        X[i, 19] = float(item_desc_len_arr[ip])

    return item_pos_list, X

In [11]:
import time
t0 = time.time()
item_pos_list, X_q = compute_features_for_query_fast(0)
t1 = time.time()
print(f"Один запрос: {t1-t0:.2f} сек., кандидатов: {len(item_pos_list)}")
print(f"Прогноз на 2452 запроса: {(t1-t0)*2452/60:.1f} мин.")

Один запрос: 0.02 сек., кандидатов: 2590
Прогноз на 2452 запроса: 1.0 мин.


In [12]:
K_FINAL = 50
predictions = []
start = time.time()

for qp in tqdm(range(len(benchmark_queries)), desc="Rerank v2"):
    item_pos_list, X_q = compute_features_for_query_fast(qp)
    scores = final_model.predict_proba(X_q)[:, 1]
    top_idx = np.argsort(-scores)[:K_FINAL]
    predictions.append([item_ids_array[item_pos_list[i]] for i in top_idx])

print(f"Инференс занял {time.time()-start:.1f} сек.")

Rerank v2: 100%|██████████| 2452/2452 [01:18<00:00, 31.28it/s]

Инференс занял 78.4 сек.


In [13]:
query_ids = benchmark_queries['query_id'].tolist()

answer_df = pd.DataFrame({
    'query_id': query_ids,
    'answer': [' '.join(p[:50]) for p in predictions],
})

# Проверки
assert (answer_df['query_id'].str.len() == 16).all()
assert answer_df['query_id'].nunique() == len(answer_df)
for ans in answer_df['answer']:
    ids = ans.split(' ')
    assert len(ids) == len(set(ids)) and len(ids) <= 50
    for iid in ids:
        assert len(iid) == 16 and iid == iid.lower()

answer_df.to_csv('answer_v2.csv', index=False)
print("Сохранено в answer_v2.csv")
print(answer_df.head(2))

Сохранено в answer_v2.csv
           query_id                                             answer
0  70DfDUpwjxB4lzFd  b92ee8f432cec2d1 255fbeaf526a1cc1 d722bcda1a55...
1  JTrdTaZJvSiLPkXj  cbeccbecb1fb8d86 367af128a9ea2a48 16ea26ff0aa0...
